# Provenance Agent — Quickstart

This notebook walks through the core functions of the provenance agent parser. Given any Jupyter notebook, the parser identifies which Python libraries are imported — the first step toward automatically generating a bibliography for the software used in a scientific workflow.

**Three functions are covered:**
- `parse_notebook` — top-level entry point: reads a `.ipynb` file and returns all imported libraries
- `extract_libraries` — lower-level utility: extracts imports from a single string of Python code
- `strip_ipython_directives` — pre-processing step: removes `%magic` and `!shell` lines that would otherwise break Python's parser

## Setup

Add `src/` to the path so the module can be imported without installing the package.

In [14]:
import sys
sys.path.insert(0, '../src')
from notebook_parser import parse_notebook, extract_libraries, strip_ipython_directives

## 1. `parse_notebook` — scan a full notebook

This is the main function you'll call. Pass a path to any `.ipynb` file and it returns the sorted list of top-level libraries imported anywhere in the notebook. Called with no argument inside a running Jupyter session, it auto-detects the current notebook path (requires `pip install ipynbname`).

In [15]:
result = parse_notebook('start_here.ipynb')
print('Libraries found:', result['libraries'])

Libraries found: ['notebook_parser', 'sys']


You can also point it at any other notebook by passing an explicit path:

```python
result = parse_notebook('sample.ipynb')
```

## 2. `extract_libraries` — parse a snippet of code

`extract_libraries` works on a raw string rather than a file. It's useful for testing or for processing individual cells. It handles both `import X` and `from X.Y import Z` forms, and always returns the top-level package name (e.g. `matplotlib.pyplot` → `matplotlib`).

In [16]:
code = """
import numpy as np
import pandas as pd
from matplotlib.pyplot import plt
"""
print(extract_libraries(code))

{'pandas', 'numpy', 'matplotlib'}


## 3. `strip_ipython_directives` — clean up magic and shell commands

Paleoclimate notebooks commonly use `%matplotlib inline`, `!pip install ...`, and similar IPython-only syntax. These lines are not valid Python and would cause `ast.parse` to fail. `strip_ipython_directives` removes them so that import extraction can proceed on the rest of the cell.

In [17]:
raw = """
%matplotlib inline
!pip install pyleoclim
import pyleoclim as pyleo
"""
print(strip_ipython_directives(raw))


import pyleoclim as pyleo


The `%` and `!` lines are dropped; the `import` line passes through unchanged. `parse_notebook` calls this automatically on every cell before parsing.

## 4. Dataset Extraction _(coming soon)_

In addition to libraries, the provenance agent will eventually identify datasets loaded in a notebook. Data in these notebooks typically comes from three sources:

- **LiPDGraph** — a graph database queried directly
- **PyLiPD** — local or web-based LiPD files
- **PANGAEA / NOAA NCEI** — datasets fetched via PyleoTUPS

For now, `parse_notebook` returns an empty list as a placeholder.

In [18]:
result = parse_notebook()
print('Datasets:', result['datasets'])  # TODO: implement dataset extraction

Datasets: []


---
## Testing on Real Notebooks

The five notebooks below are from the *Comparing Simulated and Reconstructed Climate Variability over the Past Millennium* collection. Each one loads CMIP6/PMIP model output and LMR reconstruction data via `intake`, `xarray`, and `zarr`. Running `parse_notebook` on them exercises the parser against real scientific notebooks.

### C02_b — Data Assimilation with Individual Seasonality

This notebook uses `cfr` (Climate Field Reconstruction), `numpy`, and `pickle`. It exercises the parser on a paleoclimate-specific workflow that includes serialized data loading.

In [20]:
result = parse_notebook('C02_b_DA_with_individual_seasonality.ipynb')
print(f'Libraries: {result["libraries"]}')
assert result['libraries'] == ['cfr', 'numpy', 'pickle'], f'Unexpected: {result["libraries"]}'
print('PASSED')

Libraries: ['cfr', 'numpy', 'pickle']
PASSED


In [19]:
notebooks = [
    'comparing-simulated-reconstructed-climate/CMIP6_LMR.ipynb',
    'comparing-simulated-reconstructed-climate/data_from_esm_cloudcat.ipynb',
    'comparing-simulated-reconstructed-climate/spatial_snapshots_xarray_bonuses.ipynb',
    'comparing-simulated-reconstructed-climate/VICS_dashboard.ipynb',
    'comparing-simulated-reconstructed-climate/widget_primer.ipynb',
]

for nb_path in notebooks:
    result = parse_notebook(nb_path)
    print(f'{nb_path.split("/")[-1]}')
    print(f'  Libraries: {result["libraries"]}')
    print()

CMIP6_LMR.ipynb
  Libraries: ['cartopy', 'cftime', 'collections', 'intake', 'matplotlib', 'numpy', 'os', 'pandas', 'pathlib', 'pyleoclim', 'xarray']

data_from_esm_cloudcat.ipynb
  Libraries: ['cftime', 'collections', 'intake', 'nc_time_axis', 'numpy', 'pandas', 'pyleoclim', 'xarray']

spatial_snapshots_xarray_bonuses.ipynb
  Libraries: ['cartopy', 'cftime', 'intake', 'matplotlib', 'numpy', 'pyleoclim']

VICS_dashboard.ipynb
  Libraries: ['cartopy', 'cftime', 'ipywidgets', 'matplotlib', 'numpy', 'os', 'pathlib', 'xarray']

widget_primer.ipynb
  Libraries: ['ipywidgets', 'matplotlib', 'pyleoclim', 'seaborn', 'sys', 'xarray']



<unknown>:9: SyntaxWarning: invalid escape sequence '\c'
<unknown>:3: SyntaxWarning: invalid escape sequence '\c'
<unknown>:5: SyntaxWarning: invalid escape sequence '\c'
<unknown>:6: SyntaxWarning: invalid escape sequence '\c'
<unknown>:12: SyntaxWarning: invalid escape sequence '\c'
